# 01 · Cleaning and first look at the Steam reviews

**Goal:** combine the per-game CSVs, check they cover each game's full history, clean the review text, add a few useful features, and take a first look at sentiment (thumbs up/down) over time.

**Input:** `data/raw/*_reviews.csv` (from `src/collect_reviews.py`)  
**Output:** `data/processed/reviews_clean.csv` and charts in `images/`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAW = Path("../data/raw")
OUT = Path("../data/processed")
IMG = Path("../images")
OUT.mkdir(parents=True, exist_ok=True)
IMG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

## 1. Load every game's file

In [ ]:
files = sorted(RAW.glob("*_reviews.csv"))
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(f"{len(files)} files, {len(df):,} rows")
df.groupby("game").size().rename("rows").to_frame()

## 2. Coverage check

Reviews were collected newest-first, so a game only covers its full history if the run finished. `first_month` should be close to the game's launch and `months` should have no big gaps.

In [ ]:
coverage = df.groupby("game").agg(
    reviews=("recommendationid", "size"),
    first_month=("month", "min"),
    last_month=("month", "max"),
    months=("month", "nunique"),
)
coverage

In [ ]:
# Months with suspiciously few reviews (possible gaps)
per_month = df.groupby(["game", "month"]).size().rename("n").reset_index()
per_month[per_month["n"] < 50].sort_values(["game", "month"]).head(20)

## 3. Clean the text

Each step logs how many rows it removes, so the cleaning is transparent and can go straight into the README.

In [ ]:
log = []
def step(name, before, after):
    log.append({"step": name, "rows_dropped": before - after, "rows_left": after})

n = len(df)
df = df.drop_duplicates(subset=["appid", "recommendationid"])
step("duplicate reviews (same game and id)", n, len(df)); n = len(df)

df["review"] = df["review"].astype(str).str.strip()
df = df[df["review"].str.len() > 0]
step("empty reviews", n, len(df)); n = len(df)

# Copy-paste / joke reviews: the same long text posted by several accounts.
# Only texts of 15+ words count, so short legitimate reviews like "Great game, love it" are kept.
words = df["review"].str.split().str.len()
key = df["review"].str.lower()
repeats = key.map(key.value_counts())
df = df[~((words >= 15) & (repeats >= 3))]
step("same long text posted 3+ times (copy-paste)", n, len(df)); n = len(df)

df["word_count"] = df["review"].str.split().str.len()
# Very short reviews ("10/10", "fun", "good game") are real votes. They are KEPT and flagged,
# so vote-based charts include them; text analysis (notebook 02) leaves them out.
df["is_short"] = df["word_count"] < 3

def letter_ratio(s):
    chars = [c for c in s if not c.isspace()]
    return sum(c.isalpha() for c in chars) / len(chars) if chars else 0.0

df = df[df["is_short"] | (df["review"].map(letter_ratio) >= 0.6)]
step("mostly symbols / ASCII art (3+ word reviews)", n, len(df)); n = len(df)

def ascii_share(s):
    letters = [c for c in s if c.isalpha()]
    return sum(c.isascii() for c in letters) / len(letters) if letters else 0.0

df = df[df["is_short"] | (df["review"].map(ascii_share) >= 0.9)]
step("mostly non-English characters (3+ word reviews)", n, len(df))
print("cleaning steps done, the full log is shown in section 4")

## 4. Add features

In [ ]:
df["created_utc"] = pd.to_datetime(df["created_utc"], utc=True)
df["month"] = df["created_utc"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()
df["year"] = df["month"].dt.year

df["recommended"] = df["voted_up"].astype(int)
df["char_len"] = df["review"].str.len()
df["word_count"] = df["review"].str.split().str.len()
print(f"short reviews (<3 words) kept for vote-based charts: {df['is_short'].mean():.1%}")

df["playtime_hours"] = df["playtime_at_review_min"] / 60
bins = [0, 2, 10, 50, 200, np.inf]
labels = ["<2h", "2-10h", "10-50h", "50-200h", "200h+"]
df["playtime_bucket"] = pd.cut(df["playtime_hours"], bins=bins, labels=labels, right=False)

# The newest calendar month is usually incomplete (the scrape ran mid-month),
# which makes its charts misleading. Set to False to keep it.
DROP_INCOMPLETE_LAST_MONTH = True
if DROP_INCOMPLETE_LAST_MONTH:
    latest = df["month"].max()
    n = len(df)
    df = df[df["month"] < latest]
    step(f"incomplete last month ({latest:%Y-%m})", n, len(df))

df = df.sort_values(["game", "created_utc"]).reset_index(drop=True)
df.to_csv(OUT / "reviews_clean.csv", index=False)
print(f"saved {len(df):,} cleaned reviews")
display(pd.DataFrame(log))
df[["game", "month", "recommended", "playtime_hours", "word_count", "review"]].head()

## 5. First look

In [ ]:
summary = df.groupby("game").agg(
    reviews=("recommended", "size"),
    pct_positive=("recommended", lambda s: round(100 * s.mean(), 1)),
    median_playtime_h=("playtime_hours", lambda s: round(s.median(), 1)),
    median_words=("word_count", "median"),
)
summary

### Share of positive reviews over time

In [ ]:
MIN_N = 30  # ignore months with too few reviews to be reliable
monthly = (df.groupby(["game", "month"])
             .agg(pct_positive=("recommended", "mean"), n=("recommended", "size"))
             .reset_index())
monthly = monthly[monthly["n"] >= MIN_N]
monthly["pct_positive"] *= 100

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=monthly, x="month", y="pct_positive", hue="game", marker="o", ax=ax)
ax.set(title="Share of positive reviews by month", xlabel="", ylabel="% positive", ylim=(0, 100))
ax.legend(title="")
fig.tight_layout()
fig.savefig(IMG / "positive_share_by_month.png", dpi=150)
plt.show()

### Review volume over time (after the monthly cap)

In [ ]:
vol = df.groupby(["game", "month"]).size().rename("reviews").reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=vol, x="month", y="reviews", hue="game", ax=ax)
ax.set(title="Reviews kept per month", xlabel="", ylabel="reviews")
ax.legend(title="")
fig.tight_layout()
fig.savefig(IMG / "reviews_per_month.png", dpi=150)
plt.show()

### Does playtime change how positive people are?

In [ ]:
pt = (df.dropna(subset=["playtime_bucket"])
        .pivot_table(index="game", columns="playtime_bucket", values="recommended",
                     aggfunc="mean", observed=True) * 100)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pt, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100, ax=ax,
            cbar_kws={"label": "% positive"})
ax.set(title="% positive by playtime at time of review", xlabel="", ylabel="")
fig.tight_layout()
fig.savefig(IMG / "positive_by_playtime.png", dpi=150)
plt.show()

### Review length: positive vs negative

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="game", y="word_count", hue="recommended", showfliers=False, ax=ax)
ax.set(title="Words per review", xlabel="", ylabel="words")
ax.legend(title="recommended", labels=["No", "Yes"])
plt.xticks(rotation=20)
fig.tight_layout()
fig.savefig(IMG / "review_length.png", dpi=150)
plt.show()

## Notes for the README

- Reviews are dated by when they were **first posted**, so later edits don't move them in time.
- The monthly cap keeps the newest reviews within each month, so months with more reviews than the cap are a sample, not the full count. Use volume charts as a rough guide only.
- Reviews under 3 words are kept and flagged (`is_short`) so vote-based charts include them; text analysis excludes them.
- Only English reviews were requested; an extra character check removes remaining non-English text and ASCII art from longer reviews.
- No usernames or Steam IDs are stored.